### DataFrame의 구조와 품질을 분석하는 원리를 직접 확인한 뒤, 이를 DataProfiler 클래스로 묶고 JSON 메타데이터 저장까지 검증한 노트북

In [53]:
# 프로파일링할 작은 관광 샘플 데이터 제작
from pathlib import Path
import json
import pandas as pd


sample_df = pd.DataFrame(
    {   
        # 날짜형
        "date": pd.to_datetime(
            [
                "2026-09-01",
                "2026-09-02",
                "2026-09-03",
                "2026-09-04",
                "2026-09-04",
            ]
        ),
        # 범주형
        "region": [
            "제주",
            "제주",
            "제주",
            "서귀포",
            "서귀포",
        ],
        # 수치형
        "visitors": [
            1200,
            None, # 결측값 -> NaN으로 처리
            1500,
            900,
            900,
        ],
        
        # 중복행
        "tour_type": [
            "관광지",
            "관광지",
            "축제",
            "문화시설",
            "문화시설",
        ],
    }
)

display(sample_df)

,date,region,visitors,tour_type
0,2026-09-01,제주,1200.0,관광지
1,2026-09-02,제주,NaN,관광지
2,2026-09-03,제주,1500.0,축제
3,2026-09-04,서귀포,900.0,문화시설
4,2026-09-04,서귀포,900.0,문화시설


In [ ]:
# 프로파일링이 무엇을 계산하는지 확인


# 행과 열 개수
row_count = sample_df.shape[0] # 행의 수
column_count = sample_df.shape[1] # 열의 수

print("행 수:", row_count)
print("열 수:", column_count)


# 컬럼별 자료형 확인
print("\n컬럼별 자료형:")
print(sample_df.dtypes)


# 컬럼별 결측값 개수와 비율
# 결측값이 있으면 True, 없으면 False를 반환하는 isna() 메서드 사용
missing_count = (
    sample_df
    .isna()
    .sum()
)

missing_rate = (
    sample_df
    .isna()
    .mean()
)

# 두 개의 Series를 하나의 DataFrame으로 합치기 위해 pd.DataFrame() 사용
missing_profile = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_rate": missing_rate,
    }
)

print("\n컬럼별 결측 현황:")
display(missing_profile)


# 중복 행 개수 계산
duplicate_count = int(
    sample_df
    .duplicated() # 앞에 있는 행과 동일한 행이 있으면 True, 없으면 False를 반환
    .sum()
)

print("중복 행 수:", duplicate_count)

print("\n중복으로 판정된 행:")

# 중복 행만 선택
display(
    sample_df[
        sample_df.duplicated()
    ]
)


# 수치형 컬럼 분류
numeric_columns = (
    sample_df
    .select_dtypes(
        include="number"
    )
    .columns
    .tolist() # 컬럼 이름을 리스트로 반환
)


# 날짜형 컬럼 분류
datetime_columns = (
    sample_df
    .select_dtypes(
        include="datetime"
    )
    .columns
    .tolist()
)


# 범주형 컬럼 분류
categorical_columns = (
    sample_df
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)

print("수치형 컬럼:", numeric_columns)
print("날짜형 컬럼:", datetime_columns)
print("범주형 컬럼:", categorical_columns)

행 수: 5
열 수: 4

컬럼별 자료형:
date         datetime64[ns]
region               object
visitors            float64
tour_type            object
dtype: object

컬럼별 결측 현황:


,missing_count,missing_rate
date,0,0.0
region,0,0.0
visitors,1,0.2
tour_type,0,0.0


중복 행 수: 1

중복으로 판정된 행:


,date,region,visitors,tour_type
4,2026-09-04,서귀포,900.0,문화시설


수치형 컬럼: ['visitors']
날짜형 컬럼: ['date']
범주형 컬럼: ['region', 'tour_type']


In [55]:
# 하나의 profile_stats 딕셔너리로 묶음
profile_stats = {
    "row_count": int(sample_df.shape[0]),
    "column_count": int(sample_df.shape[1]),
    "column_names": sample_df.columns.tolist(),
    "data_types": {
        column: str(dtype)
        for column, dtype in sample_df.dtypes.items()
    },
    "missing_count": {
        column: int(count)
        for column, count in sample_df.isna().sum().items()
    },
    "missing_rate": {
        column: float(rate)
        for column, rate in sample_df.isna().mean().items()
    },
    "duplicate_count": int(
        sample_df.duplicated().sum()
    ),
    "numeric_columns": (
        sample_df
        .select_dtypes(include="number")
        .columns
        .tolist()
    ),
    "datetime_columns": (
        sample_df
        .select_dtypes(include="datetime")
        .columns
        .tolist()
    ),
    "categorical_columns": (
        sample_df
        .select_dtypes(
            include=["object", "string", "category"]
        )
        .columns
        .tolist()
    ),
}

print(
    json.dumps(
        profile_stats,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "row_count": 5,
  "column_count": 4,
  "column_names": [
    "date",
    "region",
    "visitors",
    "tour_type"
  ],
  "data_types": {
    "date": "datetime64[ns]",
    "region": "object",
    "visitors": "float64",
    "tour_type": "object"
  },
  "missing_count": {
    "date": 0,
    "region": 0,
    "visitors": 1,
    "tour_type": 0
  },
  "missing_rate": {
    "date": 0.0,
    "region": 0.0,
    "visitors": 0.2,
    "tour_type": 0.0
  },
  "duplicate_count": 1,
  "numeric_columns": [
    "visitors"
  ],
  "datetime_columns": [
    "date"
  ],
  "categorical_columns": [
    "region",
    "tour_type"
  ]
}


In [56]:
# profile_stats를 바탕으로 데이터 특징을 나타내는 태그를 생성
tags = []

# 하나 이상의 컬럼에 결측값이 있는지 확인
if any(
    count > 0
    for count in profile_stats["missing_count"].values()
):
    tags.append("has_missing_values")

# 중복 행이 있는지 확인
if profile_stats["duplicate_count"] > 0:
    tags.append("has_duplicates")

# 컬럼 유형에 따른 태그 생성
if profile_stats["numeric_columns"]:
    tags.append("has_numeric_columns")

if profile_stats["datetime_columns"]:
    tags.append("has_datetime_columns")

if profile_stats["categorical_columns"]:
    tags.append("has_categorical_columns")

print("생성된 태그:")
print(tags)

생성된 태그:
['has_missing_values', 'has_duplicates', 'has_numeric_columns', 'has_datetime_columns', 'has_categorical_columns']


### ydata-profiling을 적용하기 전에, 앞에서 직접 계산한 프로파일링과 태그 생성 코드를 하나의 클래스로 묶은 수동 계산 버전

In [57]:
# 프로파일랑과 태그 생성을 dataprofiler 클래스로 묶음
class DataProfiler:

    def profile_data(self, df):
        """DataFrame의 기본 구조와 품질 정보를 계산한다."""

        if not isinstance(df, pd.DataFrame):
            raise TypeError(
                "df는 pandas DataFrame이어야 합니다."
            )

        profile_stats = {
            "row_count": int(df.shape[0]),
            "column_count": int(df.shape[1]),
            "column_names": df.columns.tolist(),
            "data_types": {
                column: str(dtype)
                for column, dtype in df.dtypes.items()
            },
            "missing_count": {
                column: int(count)
                for column, count in df.isna().sum().items()
            },
            "missing_rate": {
                column: float(rate)
                for column, rate in df.isna().mean().items()
            },
            "duplicate_count": int(
                df.duplicated().sum()
            ),
            "numeric_columns": (
                df
                .select_dtypes(include="number")
                .columns
                .tolist()
            ),
            "datetime_columns": (
                df
                .select_dtypes(include="datetime")
                .columns
                .tolist()
            ),
            "categorical_columns": (
                df
                .select_dtypes(
                    include=[
                        "object",
                        "string",
                        "category",
                    ]
                )
                .columns
                .tolist()
            ),
        }

        return profile_stats


    def generate_tags(self, profile_stats):
        """프로파일링 결과를 바탕으로 구조 태그를 생성한다."""

        tags = []

        if any(
            count > 0
            for count in profile_stats[
                "missing_count"
            ].values()
        ):
            tags.append("has_missing_values")

        if profile_stats["duplicate_count"] > 0:
            tags.append("has_duplicates")

        if profile_stats["numeric_columns"]:
            tags.append("has_numeric_columns")

        if profile_stats["datetime_columns"]:
            tags.append("has_datetime_columns")

        if profile_stats["categorical_columns"]:
            tags.append("has_categorical_columns")

        return tags


    def export_metadata(
        self,
        dataset_info,
        profile_stats,
        tags,
        out_meta_path,
    ):
        """데이터셋 정보, 프로파일 결과와 태그를 JSON으로 저장한다."""

        metadata = {
            "dataset_info": dataset_info,
            "profile_stats": profile_stats,
            "tags": tags,
        }

        output_path = Path(out_meta_path)

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with output_path.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metadata,
                file,
                ensure_ascii=False,
                indent=2,
            )

        return output_path

In [58]:
profiler = DataProfiler()

class_profile_stats = profiler.profile_data(
    sample_df
)

class_tags = profiler.generate_tags(
    class_profile_stats
)

print("클래스로 계산한 프로파일:")
print(
    json.dumps(
        class_profile_stats,
        ensure_ascii=False,
        indent=2,
    )
)

print("\n클래스로 생성한 태그:")
print(class_tags)


# 앞에서 직접 계산한 결과와 같은지 검증
assert class_profile_stats == profile_stats
assert class_tags == tags

print("\n프로파일링 및 태그 생성 검증 통과")

클래스로 계산한 프로파일:
{
  "row_count": 5,
  "column_count": 4,
  "column_names": [
    "date",
    "region",
    "visitors",
    "tour_type"
  ],
  "data_types": {
    "date": "datetime64[ns]",
    "region": "object",
    "visitors": "float64",
    "tour_type": "object"
  },
  "missing_count": {
    "date": 0,
    "region": 0,
    "visitors": 1,
    "tour_type": 0
  },
  "missing_rate": {
    "date": 0.0,
    "region": 0.0,
    "visitors": 0.2,
    "tour_type": 0.0
  },
  "duplicate_count": 1,
  "numeric_columns": [
    "visitors"
  ],
  "datetime_columns": [
    "date"
  ],
  "categorical_columns": [
    "region",
    "tour_type"
  ]
}

클래스로 생성한 태그:
['has_missing_values', 'has_duplicates', 'has_numeric_columns', 'has_datetime_columns', 'has_categorical_columns']

프로파일링 및 태그 생성 검증 통과


In [ ]:
# 설치, 불러오기 확인
try:
    from ydata_profiling import ProfileReport

    print("ydata-profiling 설치 확인 완료")

except ModuleNotFoundError:

    print("ydata-profiling이 설치되어 있지 않습니다.")

ydata-profiling 설치 확인 완료


## ydata_profiling 사용 - 실제 ydata 프로파일링 시작

ProfileReport는 DataFrame을 입력받아 다음과 같은 정보를 자동으로 계산함
- 행과 열의 개수
- 컬럼별 자료형
- 결측값 개수와 비율
- 수치형 변수의 통계
- 범주형·문자열 변수의 특징
- 전체 데이터 품질 정보

즉, 앞에서 Pandas로 직접 계산했던 프로파일링을 자동으로 수행하는 도구

In [ ]:
from ydata_profiling import ProfileReport


# 작은 샘플 데이터의 프로파일 생성
ydata_report = ProfileReport(
    sample_df,
    title="Sample Tourism Data Profile", # 프로파일 보고서의 제목
    minimal=True, #최소 모드로 프로파일링, 기본 통계 중심
)


# ydata-profiling 결과를 JSON 문자열에서 딕셔너리로 변환
ydata_profile = json.loads( # 2. JSON 문자열을 python 딕셔너리로 변환
    ydata_report.to_json() # 1. ProfileReport 객체의 분석결과를 JSON 형식의 ""문자열""로 반환
)


print("최상위 키:")
print(ydata_profile.keys())

print("\n전체 데이터 통계:")
print(
    json.dumps(
        ydata_profile["table"],
        ensure_ascii=False,
        indent=2,
    )
)

print("\n컬럼별 프로파일 대상:")
print(
    list(
        ydata_profile["variables"].keys()
    )
)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 386.39it/s]


Render JSON:   0%|          | 0/1 [00:00<?, ?it/s]

최상위 키:
dict_keys(['analysis', 'time_index_analysis', 'table', 'variables', 'scatter', 'correlations', 'missing', 'alerts', 'package', 'sample', 'duplicates'])

전체 데이터 통계:
{
  "n": 5,
  "n_var": 4,
  "memory_size": 292,
  "record_size": 58.4,
  "n_cells_missing": 1,
  "n_vars_with_missing": 1,
  "n_vars_all_missing": 0,
  "p_cells_missing": 0.05,
  "types": {
    "DateTime": 1,
    "Text": 2,
    "Numeric": 1
  }
}

컬럼별 프로파일 대상:
['date', 'region', 'visitors', 'tour_type']


키	            : 내용
* analysis :	    프로파일 분석 시간 등 기본 정보
* table :	        DataFrame 전체 통계
* variables	:       각 컬럼의 상세 통계
* correlations : 	컬럼 간 상관관계
* missing :	        결측값 관련 분석
* alerts :	        데이터 품질 경고
* sample :	        데이터 일부 샘플
* duplicates :	    중복 관련 결과

항목	: 의미
* n :	행 수
* n_var :	컬럼 수
* memory_size :	DataFrame이 사용하는 메모리 크기
* record_size :	행 하나당 평균 메모리 크기
* n_cells_missing :	전체 결측 셀 개수
* n_vars_with_missing :	결측값을 가진 컬럼 수
* n_vars_all_missing :	모든 값이 결측인 컬럼 수
* p_cells_missing	: 전체 셀 기준 결측률
* types :	ydata가 판별한 자료형별 컬럼 수

In [ ]:
# 컬럼별 결측률과 자료형, 중복 결과가 JSON 구조로 포함되어 있는지 확인
print("컬럼별 핵심 통계:")

# variables 딕셔너리에 들어 있는 컬럼별 프로파일 결과를 반복문으로 출력
for column, stats in ydata_profile[
    "variables"
].items():
    # 필요한 정보만 추출하여 출력
    print(
        {
            "column": column,
            "type": stats.get("type"), # ydata-profiling에서 제공하는 컬럼 자료형
            "missing_count": stats.get("n_missing"),
            "missing_rate": stats.get("p_missing"),
        }
    )


print("\n중복 관련 결과:")
print(
    json.dumps(
        ydata_profile.get("duplicates"), # ydata-profiling에서 제공하는 중복(duplicates 키 값) 관련 통계
        ensure_ascii=False,
        indent=2,
    )
)

컬럼별 핵심 통계:
{'column': 'date', 'type': 'DateTime', 'missing_count': 0, 'missing_rate': 0.0}
{'column': 'region', 'type': 'Text', 'missing_count': 0, 'missing_rate': 0.0}
{'column': 'visitors', 'type': 'Numeric', 'missing_count': 1, 'missing_rate': 0.2}
{'column': 'tour_type', 'type': 'Text', 'missing_count': 0, 'missing_rate': 0.0}

중복 관련 결과:
"None"


### ydata-profiling의 컬럼별 분석 결과에서 자료형과 결측 정보를 추출하고, Pandas로 중복 개수를 보완하여 태그 생성과 JSON 저장에 사용할 간단한 profile_stats 딕셔너리로 재구성

In [ ]:
# ydata-profiling 결과에서 필요한 정보만 추출하여 profile_stats 딕셔너리로 변환
variable_profiles = ydata_profile["variables"]

# 수치형 컬럼 추출
numeric_columns = [
    column
    for column, stats in variable_profiles.items()
    if stats.get("type") == "Numeric"
]

# 날짜형 컬럼 추출
datetime_columns = [
    column
    for column, stats in variable_profiles.items()
    if stats.get("type") == "DateTime"
]

# 범주형 컬럼 추출
categorical_columns = [
    column
    for column, stats in variable_profiles.items()
    # 세 종류 중 하나라도 해당하면 범주형으로 분류
    if stats.get("type") in {
        "Text",
        "Categorical",
        "Boolean",
    }
]

# ydata-profiling 결과를 바탕으로 profile_stats 딕셔너리 생성
ydata_profile_stats = {
    #행 수
    "row_count": int(
        ydata_profile["table"]["n"]
    ),
    #열 수
    "column_count": int(
        ydata_profile["table"]["n_var"]
    ),
    #컬럼명 목록
    "column_names": list(
        variable_profiles.keys()
    ),
    #컬럼별 자료형
    "data_types": {
        column: stats.get("type")
        for column, stats in variable_profiles.items()
    },
    #컬럼별 결측값 개수
    "missing_count": {
        column: int(
            stats.get("n_missing", 0) # 없으면 0 반환
        )
        for column, stats in variable_profiles.items()
    },
    #컬럼별 결측값 비율
    "missing_rate": {
        column: float(
            stats.get("p_missing", 0.0) # 없으면 0.0 반환
        )
        for column, stats in variable_profiles.items()
    },
    #전체 결측값 비율
    "overall_missing_rate": float(
        ydata_profile["table"]["p_cells_missing"]
    ),
    #중복 행 개수
    "duplicate_count": int(
        sample_df.duplicated().sum()
    ),
    #수치형 컬럼 목록
    "numeric_columns": numeric_columns,
    "numeric_column_count": len(
        numeric_columns
    ),
    #날짜형 컬럼 목록
    "datetime_columns": datetime_columns,
    "has_datetime": bool(
        datetime_columns
    ),
    #범주형 컬럼 목록
    "categorical_columns": categorical_columns,
    "categorical_column_count": len(
        categorical_columns
    ),
}


print(
    json.dumps(
        ydata_profile_stats,
        ensure_ascii=False,
        indent=2,
    )
)


{
  "row_count": 5,
  "column_count": 4,
  "column_names": [
    "date",
    "region",
    "visitors",
    "tour_type"
  ],
  "data_types": {
    "date": "DateTime",
    "region": "Text",
    "visitors": "Numeric",
    "tour_type": "Text"
  },
  "missing_count": {
    "date": 0,
    "region": 0,
    "visitors": 1,
    "tour_type": 0
  },
  "missing_rate": {
    "date": 0.0,
    "region": 0.0,
    "visitors": 0.2,
    "tour_type": 0.0
  },
  "overall_missing_rate": 0.05,
  "duplicate_count": 1,
  "numeric_columns": [
    "visitors"
  ],
  "numeric_column_count": 1,
  "datetime_columns": [
    "date"
  ],
  "has_datetime": true,
  "categorical_columns": [
    "region",
    "tour_type"
  ],
  "categorical_column_count": 2
}


## export_metadata()가 JSON 파일을 제대로 저장하는지 검증

In [ ]:
profiler = DataProfiler()

# sample_df를 profile_data() 메서드에 전달
stats = profiler.profile_data(
    sample_df
)

# 앞에서 만든 프로파일링 결과 stats를 generate_tags()에 전달
# generate_tags()는 stats의 내용을 확인하여 조건에 맞는 태그를 만듦
tags = profiler.generate_tags(
    stats
)

print("생성된 태그:", tags)

생성된 태그: ['has_missing_values', 'has_duplicates', 'has_numeric_columns', 'has_datetime_columns', 'has_categorical_columns']


### 지금까지 만든 데이터셋 기본 정보, 프로파일링 결과, 태그를 하나의 JSON 파일로 저장

In [ ]:
# 데이터셋 기본 정보 만들기
dataset_info = {
    "id": "TOUR_SAMPLE_01",
    "name": "관광 샘플 데이터",
}

# 메타데이터 파일로 저장
metadata_output_path = profiler.export_metadata(
    dataset_info=dataset_info,
    profile_stats=stats,
    tags=tags,
    out_meta_path=(
        "outputs/"
        "sample_tourism_metadata.json"
    ),
)

print(
    "메타데이터 저장 경로:",
    metadata_output_path,
)

메타데이터 저장 경로: outputs\sample_tourism_metadata.json


### 검증

In [ ]:
# JSON 파일 다시 열기
with metadata_output_path.open(
    "r",
    encoding="utf-8",
) as file:
    # JSON 파일을 읽어서 딕셔너리로 변환
    saved_metadata = json.load(file)

# 데이터셋 기본 정보 검증
assert saved_metadata[
    "dataset_info"
] == dataset_info

# 프로파일링 결과 검증
assert saved_metadata[
    "profile_stats"
] == stats

# 태그 검증
assert saved_metadata[
    "tags"
] == tags

# 검증에서 True면 정상적으로 저장, False면 저장 실패

print("메타데이터 JSON 저장 검증 통과")

메타데이터 JSON 저장 검증 통과
